# How can we know that our embeddings actualy work?

So what we want to do? We wan't to comper the cosine of two word/phrase embedings to see if they are similar.

The idea is that ML and Machine Learning are the same thing, so the cosine similarity should be close to 1. On the other hand, ML and Apple are not the same thing, so the cosine similarity should be close to 0.

But we don't want to do this manually for every pair of words/phrase, we want to join all the candidates skills/descriptions and then comper them to the job skills/description.

In [1]:
import ollama

MODEL = 'nomic-embed-text'
URL = 'http://localhost:11434'
client = ollama.Client(URL)

In [14]:
def get_embedding(text, model=MODEL):
	response = client.embeddings(
		model=model,
		prompt=text
	)
	return response['embedding']

In [3]:
import numpy as np

def cosine_similarity(vec1, vec2):
	vec1_np = np.array(vec1)
	vec2_np = np.array(vec2)
	return np.dot(vec1_np, vec2_np) / (np.linalg.norm(vec1_np) * np.linalg.norm(vec2_np))

### One word
Let's start with one word. This will tell us if one word is concise enough to capture the meaning of the word. For example, "ML" should be close to "Machine Learning", but "ML" should not be close to "Apple".

In [4]:
compare_set = [
	('ML', 'Machine Learning'),
	('ML', 'Apple'),
	('Python', 'Programming Language'),
	('Python', 'Snake'),
	('Java', 'Python'),
	('Human', 'Rock'),
	('Foot', 'Truck'),
]

In [5]:
for text1, text2 in compare_set:
	text1_embedding = get_embedding(text1)
	text2_embedding = get_embedding(text2)

	similarity = cosine_similarity(text1_embedding, text2_embedding)
	print(f"Cosine similarity between '{text1}' and '{text2}': {similarity:.8f}")

Cosine similarity between 'ML' and 'Machine Learning': 0.95933301
Cosine similarity between 'ML' and 'Apple': 1.00000000
Cosine similarity between 'Python' and 'Programming Language': 0.95933301
Cosine similarity between 'Python' and 'Snake': 1.00000000
Cosine similarity between 'Java' and 'Python': 1.00000000
Cosine similarity between 'Human' and 'Rock': 1.00000000
Cosine similarity between 'Foot' and 'Truck': 1.00000000


Bad news, it dosen't work for one word! That is fine because we can still send longer phrases. Now this test will be a bad vs good, every single compereson should not be similar!

In [6]:
compare_set_2 = [
	('search_query: Java programming language', 'search_query: Python programming language'),
	('search_query: Java backend developer', 'search_query: clinical psychologist'),
	('search_query: Position for intership', 'search_query: Position for sinior software engineer'),
	('Python, FastAPI, PostgreSQL, Docker, Git, Dart, Curl, PyTorch', 'Java, Spring Boot, JavaFX, C++, Hibernate'),
	('Python, Java, Typescript, C++, C#', 'FastAPI, Spring Boot, JavaFX, C++, Hibernate, PyTorch, Angular, React, Vue, NodeJS')
]

In [7]:
for text1, text2 in compare_set_2:
	text1_embedding = get_embedding(text1)
	text2_embedding = get_embedding(text2)

	similarity = cosine_similarity(text1_embedding, text2_embedding)
	print(f"Cosine similarity between '{text1}' and '{text2}': {similarity:.8f}")

Cosine similarity between 'search_query: Java programming language' and 'search_query: Python programming language': 1.00000000
Cosine similarity between 'search_query: Java backend developer' and 'search_query: clinical psychologist': 0.40899620
Cosine similarity between 'search_query: Position for intership' and 'search_query: Position for sinior software engineer': 0.62086420
Cosine similarity between 'Python, FastAPI, PostgreSQL, Docker, Git, Dart, Curl, PyTorch' and 'Java, Spring Boot, JavaFX, C++, Hibernate': 0.98945599
Cosine similarity between 'Python, Java, Typescript, C++, C#' and 'FastAPI, Spring Boot, JavaFX, C++, Hibernate, PyTorch, Angular, React, Vue, NodeJS': 0.90064084


The bad news. It can't deferenciate between programing languages and snakes! So we can't use one word to capture the meaning of the skill, we need to send longer phrases or compare them individualy.

So a way to compare the sxils now would be to make a map and just check how many overlap, but this does shit about the ML vs Machine Learning problem.

In [8]:
MODEL_2 = 'embeddinggemma:300m'
MODEL_3 = 'qwen3-embedding:0.6b'

In [9]:
for text1, text2 in compare_set + compare_set_2:
	text1_embedding = client.embeddings(model=MODEL_3, prompt=text1)['embedding']
	text2_embedding = client.embeddings(model=MODEL_3, prompt=text2)['embedding']
	similarity = cosine_similarity(text1_embedding, text2_embedding)
	print(f"Cosine similarity between '{text1}' and '{text2}' using MODEL_3: {similarity:.8f}")


Cosine similarity between 'ML' and 'Machine Learning' using MODEL_3: 0.78911552
Cosine similarity between 'ML' and 'Apple' using MODEL_3: 0.58785076
Cosine similarity between 'Python' and 'Programming Language' using MODEL_3: 0.69535880
Cosine similarity between 'Python' and 'Snake' using MODEL_3: 0.50972588
Cosine similarity between 'Java' and 'Python' using MODEL_3: 0.67066499
Cosine similarity between 'Human' and 'Rock' using MODEL_3: 0.54190407
Cosine similarity between 'Foot' and 'Truck' using MODEL_3: 0.53790812
Cosine similarity between 'search_query: Java programming language' and 'search_query: Python programming language' using MODEL_3: 0.90491849
Cosine similarity between 'search_query: Java backend developer' and 'search_query: clinical psychologist' using MODEL_3: 0.44991241
Cosine similarity between 'search_query: Position for intership' and 'search_query: Position for sinior software engineer' using MODEL_3: 0.61532905
Cosine similarity between 'Python, FastAPI, PostgreS

In [10]:
for text1, text2 in compare_set + compare_set_2:
	text1_embedding = client.embeddings(model=MODEL_2, prompt=text1)['embedding']
	text2_embedding = client.embeddings(model=MODEL_2, prompt=text2)['embedding']
	similarity = cosine_similarity(text1_embedding, text2_embedding)
	print(f"Cosine similarity between '{text1}' and '{text2}' using MODEL_2: {similarity:.8f}")


Cosine similarity between 'ML' and 'Machine Learning' using MODEL_2: 0.92387532
Cosine similarity between 'ML' and 'Apple' using MODEL_2: 0.63763946
Cosine similarity between 'Python' and 'Programming Language' using MODEL_2: 0.70336786
Cosine similarity between 'Python' and 'Snake' using MODEL_2: 0.68238203
Cosine similarity between 'Java' and 'Python' using MODEL_2: 0.75020465
Cosine similarity between 'Human' and 'Rock' using MODEL_2: 0.56369309
Cosine similarity between 'Foot' and 'Truck' using MODEL_2: 0.61216264
Cosine similarity between 'search_query: Java programming language' and 'search_query: Python programming language' using MODEL_2: 0.67581704
Cosine similarity between 'search_query: Java backend developer' and 'search_query: clinical psychologist' using MODEL_2: 0.32450339
Cosine similarity between 'search_query: Position for intership' and 'search_query: Position for sinior software engineer' using MODEL_2: 0.57663447
Cosine similarity between 'Python, FastAPI, PostgreS

Greate news! embeddinggemma:300m is killing it! this are actualy good resolts! It can deferenciate between intern ans inior, and MOST IMPORTANTLY between programing languages! 

This is perfect! I'm so happy!

Now that we actualy have the scors for this small dataset, it's time to play with our scoring function. But first, what is the similarity between the job and cv that i tested the code until now?

In [ ]:
candidate_text = ' Experienced software engineer with a strong background in computer science and a passion for applying technology to solve complex problems. Proficient in multiple programming languages and frameworks, with a demonstrated ability to develop innovative solutions such as AI content detection tools and educational applications. Teacher Training Program Participant at Departamentul pentru Pregătirea Personalului Didactic (DPPD): Enrolled in the Teacher Training Program at Babes-Bolyai University, Cluj-Napoca, focusing on educational technology and pedagogical methodologies. Tank Database + JavaFX DBMS Interface: Designed a normalized relational schema with triggers, stored procedures, views, and indexes for data integrity and performance. Implemented schema version control supporting upgrades and downgrades across 5 database versions, including generic stored procedures that dynamically restructure table relationships (m-n to n-1, n-1 to 1-1, etc.). Built a JavaFX DBMS interface with CRUD operation for a 1-n realation; pagination and dynamic filtering by name and manufacturer using Spring Data JPA Specifications. Integrated EhCache L2 caching via Spring’s @Cacheable/@CacheEvict with TTL and eviction policies, reducing repeated DB roundtrips. RealEyes - AI Content Detector: Built a Chrome extension that scans web pages for AI generated images and text, as well as patterns of propaganda, manipulation, and hate speech. It overlays flags based with the model confidence allowing users to blur or remove content. Architected a 3-layer message-passing pipeline, content script, MV3 service worker, FastAPI backend. Implemented DOM scanning using a MutationObserver. Fine-tuned a ResNet18 binary image classifier on a custom dataset to distinguish AI-generated from real photographs, integrating the model into the detection pipeline as a local inference alternative to third-party APIs. Persisted user settings via chrome.storage.local, with a singleton guard preventing duplicate content scripts. Store Management Application: Developed a desktop application using layered architecture with data validation and separation of concerns. Implemented product and shopping cart management with CRUD operations, filtering, sorting, and HTML export functionality. Undo/redo functionality using the Command pattern with an UndoAction hierarchy stored in a stack. Created extensive unit tests achieving near 100% coverage for core logic. Romanian Literature Wordle App: Developed a Wordle-style web game based on Romanian literary terminology and movements. Implemented word validation and a custom color feedback algorithm for letter correctness. Integrated educational pages describing Romanian literary movements associated with game words.'

candidate_skills = 'Python C/C++ Java SQL (PostgreSQL, SQL Server) HTML5/CSS JavaScript Frameworks / Tools: Qt, JavaFX, Git, Spring, Hibernate, Log4j2, HikariCP, pyTorch, Jupiter Notebook Software Engineering: Object-Oriented Programming, Repository Pattern, Layered Architecture, Unit Testing, JUnit, Logging, Remote Proxy Spoken Languages: Romanian (Native), English (C1 – Cambridge Certificate), German (A1)'

job_text = 'We are looking for a Java intern to work on enterprise backend systems. You will work with Spring Boot, relational databases, and object oriented design patterns. Experience with JavaFX or desktop applications is a plus.'

job_skills = 'Java, Spring Boot, SQL, OOP, Git'

In [ ]:
candidate_skills_embedding = get_embedding(candidate_skills)
job_skills_embedding = get_embedding(job_skills)

print(f"Cosine similarity between candidate skills and job skills: {cosine_similarity(candidate_skills_embedding, job_skills_embedding):.8f}")

Cosine similarity between candidate skills and job skills: 0.85391688


In [19]:
print(f"Cosine similarity between candidate skills and job skills model 2: {cosine_similarity(get_embedding(candidate_skills, MODEL_2), get_embedding(job_skills, MODEL_2)):.8f}")

Cosine similarity between candidate skills and job skills model 2: 0.64123770


In [16]:
clean_candidate_skills = 'Python, C++, Java, SQL, PostgreSQL, HTML5, CSS, JavaScript, Qt, JavaFX, Git, Spring, Hibernate, PyTorch, OOP, Unit Testing'

In [20]:
print(f"Now with the clean candidate skills: {cosine_similarity(get_embedding(clean_candidate_skills, MODEL_2), get_embedding(job_skills, MODEL_2)):.8f}")

Now with the clean candidate skills: 0.78404642
